# 1. Setup Paths and Data Sources 

Define imports, workspace paths, and the source locations.

In [1]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

DAIOE_SOURCE: str = (
    "https://raw.githubusercontent.com/joseph-data/07_translate_ssyk/main/"
    "03_translated_files/daioe_ssyk2012_translated.csv"
)
SCB_SOURCE: str = (
    "https://raw.githubusercontent.com/joseph-data/AI_Econ_daioe_years_regions/daioe_pull/"
    "data/processed/ssyk12_aggregated_ssyk4_to_ssyk1.parquet"
)



# 2. Load DAIOE and SCB Lazily

Read source files as `LazyFrame` objects for efficient transformations

In [2]:
daioe_lazy_lf = pl.scan_csv(
    DAIOE_SOURCE,
)

scb_lazy_lf = pl.scan_parquet(
    SCB_SOURCE,
)

# 3. Quick Sanity Checks

Preview data 

In [3]:
print(daioe_lazy_lf.head(5).collect())

shape: (5, 27)
┌────────────┬──────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ ssyk2012_4 ┆ year ┆ daioe_alla ┆ daioe_stra ┆ … ┆ pctl_rank_ ┆ ssyk2012_ ┆ ssyk2012_ ┆ ssyk2012_ │
│ ---        ┆ ---  ┆ pps        ┆ tgames     ┆   ┆ genai      ┆ 1         ┆ 2         ┆ 3         │
│ str        ┆ i64  ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆      ┆ f64        ┆ f64        ┆   ┆ f64        ┆ str       ┆ str       ┆ str       │
╞════════════╪══════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 0110 Commi ┆ 2010 ┆ null       ┆ null       ┆ … ┆ null       ┆ 0 Armed   ┆ 01        ┆ 011 Commi │
│ ssioned    ┆      ┆            ┆            ┆   ┆            ┆ forces    ┆ Officers  ┆ ssioned   │
│ armed      ┆      ┆            ┆            ┆   ┆            ┆ occupatio ┆           ┆ armed     │
│ forces…    ┆      ┆            ┆            ┆   ┆            ┆ ns        ┆

In [4]:
print(scb_lazy_lf.head(5).collect())

shape: (5, 8)
┌───────┬───────────┬───────────────────────┬─────────────┬─────────────────┬───────┬──────┬───────┐
│ level ┆ ssyk_code ┆ occupation            ┆ county_code ┆ county          ┆ sex   ┆ year ┆ value │
│ ---   ┆ ---       ┆ ---                   ┆ ---         ┆ ---             ┆ ---   ┆ ---  ┆ ---   │
│ str   ┆ str       ┆ str                   ┆ str         ┆ str             ┆ str   ┆ i64  ┆ i64   │
╞═══════╪═══════════╪═══════════════════════╪═════════════╪═════════════════╪═══════╪══════╪═══════╡
│ SSYK4 ┆ 3324      ┆ Orders coordinator    ┆ 03          ┆ Uppsala county  ┆ women ┆ 2023 ┆ 201   │
│ SSYK4 ┆ 4321      ┆ Warehouse and         ┆ 03          ┆ Uppsala county  ┆ women ┆ 2023 ┆ 54    │
│       ┆           ┆ terminal supervi…     ┆             ┆                 ┆       ┆      ┆       │
│ SSYK4 ┆ 3321      ┆ Insurance sellers and ┆ 12          ┆ Skåne county    ┆ men   ┆ 2016 ┆ 662   │
│       ┆           ┆ insuranc…             ┆             ┆                 ┆

# 4. Compute the 1-Yr, 3-Yr and 5-Yr Change

Aggregate to employment totals by `level`, `ssyk_code`, `occupation`, `county_code`, `sex`, and `year`, then compute absolute and percent changes over 1, 3, and 5 years using window shifts.

In [5]:
change_keys = [col for col in scb_lazy_lf.collect_schema().names() if col != "value"]
time_col = "year"
group_keys = [col for col in change_keys if col != time_col]

scb_lazy_lf_changes = (
    scb_lazy_lf
    .group_by(change_keys)
    .agg(pl.col("value").sum().alias("emp_count"))
    # Pre-compute shifted values once each
    .with_columns([
        pl.col("emp_count").shift(1).over(group_keys, order_by=time_col).alias("_emp_1y"),
        pl.col("emp_count").shift(3).over(group_keys, order_by=time_col).alias("_emp_3y"),
        pl.col("emp_count").shift(5).over(group_keys, order_by=time_col).alias("_emp_5y"),
    ])
    # Reuse pre-computed shifts
    .with_columns([
        (pl.col("emp_count") - pl.col("_emp_1y")).alias("chg_1y"),
        (pl.col("emp_count") - pl.col("_emp_3y")).alias("chg_3y"),
        (pl.col("emp_count") - pl.col("_emp_5y")).alias("chg_5y"),
        ((pl.col("emp_count") / pl.col("_emp_1y") - 1) * 100).alias("pct_chg_1y"),
        ((pl.col("emp_count") / pl.col("_emp_3y") - 1) * 100).alias("pct_chg_3y"),
        ((pl.col("emp_count") / pl.col("_emp_5y") - 1) * 100).alias("pct_chg_5y"),
    ])
    .drop("_emp_1y", "_emp_3y", "_emp_5y")
)

scb_lazy_lf_changes.collect_schema()

Schema([('level', String),
        ('ssyk_code', String),
        ('occupation', String),
        ('county_code', String),
        ('county', String),
        ('sex', String),
        ('year', Int64),
        ('emp_count', Int64),
        ('chg_1y', Int64),
        ('chg_3y', Int64),
        ('chg_5y', Int64),
        ('pct_chg_1y', Float64),
        ('pct_chg_3y', Float64),
        ('pct_chg_5y', Float64)])

# 5. Derive SSYK Levels in DAIOE

Split DAIOE SSYK4 into SSYK1-4 and keep SSYK2012-era years.

In [6]:
SSYK12_START_YEAR = 2014  # First year of SSYK2012 publication

daioe_lazy_lf_ssyk12 = (
    daioe_lazy_lf
    .with_columns(
        pl.col("ssyk2012_4").str.slice(0, i).alias(f"code_{i}")
        for i in range(1, 5)
    )
    .drop(pl.col("^ssyk2012.*$"))
    .filter(pl.col("year") >= SSYK12_START_YEAR)
)
daioe_lazy_lf_ssyk12.limit(10).collect()


year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
2014,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2015,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2016,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2017,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2018,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2019,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2020,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2021,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2022,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""


# 6. Align DAIOE Years to SCB Coverage

Extend DAIOE series forward when SCB has later years.

In [7]:
## Here I extend the years to Latest according to the pulled SCB data (2024, yearly)

base = daioe_lazy_lf_ssyk12

# Extend years to latest available in SCB data (2024, yearly)
DAIOE_MAX_YEAR = base.select(pl.max("year")).collect().item()
SCB_MAX_YEAR = scb_lazy_lf.select(pl.max("year")).collect().item()
missing_years = list(range(DAIOE_MAX_YEAR + 1, SCB_MAX_YEAR + 1))

daioe_lazy_lf_extended = (
    base
    if not missing_years
    else pl.concat(
        [
            base,
            base
            .filter(pl.col("year") == DAIOE_MAX_YEAR)
            .drop("year")
            .join(pl.LazyFrame({"year": missing_years}), how="cross")
            .select(base.collect_schema().names()),
        ],
        how="vertical",
    )
)

In [8]:
daioe_lazy_lf_extended.filter(pl.col("year") == SCB_MAX_YEAR).limit(10).collect()

year,daioe_allapps,daioe_stratgames,daioe_videogames,daioe_imgrec,daioe_imgcompr,daioe_imggen,daioe_readcompr,daioe_lngmod,daioe_translat,daioe_speechrec,daioe_genai,pctl_rank_allapps,pctl_rank_stratgames,pctl_rank_videogames,pctl_rank_imgrec,pctl_rank_imgcompr,pctl_rank_imggen,pctl_rank_readcompr,pctl_rank_lngmod,pctl_rank_translat,pctl_rank_speechrec,pctl_rank_genai,code_1,code_2,code_3,code_4
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""01""","""011""","""0110"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""02""","""021""","""0210"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""0""","""03""","""031""","""0310"""
2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""1""","""11""","""111""","""1111"""
2024,31.355038,0.25465,3.619455,0.402864,0.224535,0.739058,0.423185,1.027839,0.040984,0.735677,2.899789,47.869999,42.200001,7.21,42.91,54.02,62.77,59.93,57.330002,56.619999,55.439999,62.529999,"""1""","""11""","""111""","""1112"""
2024,33.85648,0.269727,3.7854,0.439125,0.247134,0.788555,0.461196,1.130982,0.04624,0.819611,3.146769,59.93,58.040001,11.7,58.509998,65.370003,72.93,68.910004,68.440002,70.330002,66.550003,70.57,"""1""","""11""","""111""","""1113"""
2024,32.331161,0.260123,3.675521,0.405983,0.23089,0.748559,0.441446,1.077137,0.043876,0.78386,2.993233,51.889999,50.0,8.16,44.330002,56.380001,64.660004,65.599998,63.950001,62.77,58.98,64.889999,"""1""","""11""","""112""","""1120"""
2024,38.505562,0.333748,4.106049,0.511271,0.285375,0.847376,0.551517,1.357987,0.05216,0.922529,3.600155,82.150002,89.239998,21.870001,81.68,84.989998,82.389999,87.830002,88.529999,84.75,82.620003,86.169998,"""1""","""12""","""121""","""1211"""
2024,38.505562,0.333748,4.106049,0.511271,0.285375,0.847376,0.551517,1.357987,0.05216,0.922529,3.600155,82.389999,89.010002,22.1,81.910004,84.75,82.620003,87.589996,88.769997,84.519997,82.389999,85.93,"""1""","""12""","""121""","""1212"""
